# 03 — ζ(s) et les nombres premiers

In [6]:
import numpy as np
import mpmath
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.special import expi
from sympy import primepi, primerange

mpmath.mp.dps = 25

TEMPLATE     = "plotly_dark"
COLOR_MAIN   = "#7C6FCD"
COLOR_ACCENT = "#F2A623"
COLOR_MUTED  = "#888780"
COLOR_DANGER = "#E24B4A"
COLOR_TEAL   = "#1D9E75"

## Le théorème des nombres premiers (TNP)

On note $\pi(x)$ le nombre de premiers $\leq x$. Le TNP dit :

$$\pi(x) \sim \frac{x}{\ln x} \quad (x \to \infty)$$

Gauss l'avait deviné empiriquement en comptant les premiers dans des tables.  
Il a remarqué que la **densité locale** des premiers autour de $x$ vaut $\frac{1}{\ln x}$ — d'où l'intégrale :

$$\text{Li}(x) = \int_2^x \frac{dt}{\ln t}$$

Beaucoup plus précise que $x/\ln x$ — erreur < 0.2% contre ~8% pour $x = 10^6$.

In [7]:
def Li(x):
    return expi(np.log(x)) - expi(np.log(2))

# Comparaison sur x = 10^6
x = 1_000_000
exact = int(primepi(x))
approx_log = x / np.log(x)
approx_li  = Li(x)

print(f"π({x:,})       = {exact:,}")
print(f"x/ln(x)        = {approx_log:,.0f}  — erreur {abs(exact-approx_log)/exact*100:.2f}%")
print(f"Li(x)          = {approx_li:,.0f}  — erreur {abs(exact-approx_li)/exact*100:.4f}%")

# Visualisation sur [2, 1000]
x_vals   = np.arange(2, 1001)
pi_exact = [int(primepi(x)) for x in x_vals]
li_vals  = [Li(x) for x in x_vals]
log_vals = [x / np.log(x) for x in x_vals]

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_vals, y=pi_exact, name="π(x) exact",
    line=dict(color=COLOR_ACCENT, width=2)))
fig.add_trace(go.Scatter(x=x_vals, y=li_vals, name="Li(x) — Gauss",
    line=dict(color=COLOR_TEAL, width=1.5)))
fig.add_trace(go.Scatter(x=x_vals, y=log_vals, name="x/ln(x) — TNP",
    line=dict(color=COLOR_MUTED, width=1.5, dash="dash")))

fig.update_layout(template=TEMPLATE, height=440,
    title="π(x) — TNP et approximation de Gauss",
    xaxis_title="x", yaxis_title="Nombre de premiers ≤ x",
    hovermode="x unified")
fig.show()

π(1,000,000)       = 78,498
x/ln(x)        = 72,382  — erreur 7.79%
Li(x)          = 78,627  — erreur 0.1637%


## Le produit d'Euler — ζ encodée par les premiers

Euler (1737) découvre que la même fonction ζ(s) s'écrit de deux façons :

$$\sum_{n=1}^{\infty} \frac{1}{n^s} = \prod_{p \text{ premier}} \frac{1}{1-p^{-s}}$$

Pourquoi ? Parce que tout entier se décompose **uniquement** en produit de premiers.  
Sommer sur tous les entiers revient à sommer sur toutes les combinaisons de premiers.

**Conséquence** : ζ(s) contient toute l'information sur les premiers.

In [8]:
from sympy import primerange

def euler_product(s, n_primes=1000):
    """Produit d'Euler tronqué aux n_primes premiers premiers."""
    primes = list(primerange(2, 8000))[:n_primes]
    return np.prod([1 / (1 - p**(-s)) for p in primes])

print("Vérification du produit d'Euler :")
print(f"{'s':>4}  {'Produit Euler':>16}  {'ζ(s) exact':>16}  {'Erreur':>10}")
print("-" * 52)
for s in [2, 3, 4, 5, 10]:
    prod  = euler_product(s)
    exact = float(mpmath.zeta(s))
    print(f"  {s:>2}  {prod:>16.8f}  {exact:>16.8f}  {abs(prod-exact):>10.2e}")

Vérification du produit d'Euler :
   s     Produit Euler        ζ(s) exact      Erreur
----------------------------------------------------
   2        1.64491317        1.64493407    2.09e-05
   3        1.20205690        1.20205690    1.01e-09
   4        1.08232323        1.08232323    3.31e-14
   5        1.03692776        1.03692776    1.20e-14
  10        1.00099458        1.00099458    8.88e-16


## Les zéros non triviaux et π(x)

Riemann (1859) montre que π(x) s'exprime **exactement** via les zéros de ζ :

$$\pi(x) = \text{Li}(x) - \sum_{\rho} \text{Li}(x^\rho) + \text{termes mineurs}$$

où la somme porte sur tous les zéros non triviaux $\rho = \frac{1}{2} + it$.

Chaque zéro ajoute une **correction oscillante** — comme une décomposition de Fourier.  
Li(x) surestime toujours légèrement π(x) — c'est l'écart qu'on visualise ci-dessous.  
**L'hypothèse de Riemann** garantit que cet écart reste borné par $O(\sqrt{x} \ln x)$.

In [9]:
# Les 20 premiers zéros non triviaux
zeros_t = [float(mpmath.zetazero(n).imag) for n in range(1, 21)]
print("20 premiers zéros (partie imaginaire) :")
print([round(t, 3) for t in zeros_t])

# Visualisation : |ζ(½+it)| — les annulations sont les zéros
t_line   = np.linspace(0.1, 80, 3000)
zeta_mod = [abs(complex(mpmath.zeta(0.5 + 1j*t))) for t in t_line]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=t_line, y=zeta_mod,
    name="|ζ(½+it)|",
    line=dict(color=COLOR_MAIN, width=1.5),
    fill="tozeroy", fillcolor="rgba(124,111,205,0.1)"
))
fig.add_trace(go.Scatter(
    x=zeros_t, y=[0]*len(zeros_t),
    mode="markers",
    marker=dict(color=COLOR_DANGER, size=10),
    name="Zéros non triviaux"
))
fig.add_hline(y=0, line_color=COLOR_MUTED, opacity=0.3)
fig.update_layout(
    template=TEMPLATE, height=400,
    title="|ζ(½+it)| — chaque annulation est un zéro non trivial",
    xaxis_title="t = Im(s)", yaxis_title="|ζ(½+it)|"
)
fig.show()

20 premiers zéros (partie imaginaire) :
[14.135, 21.022, 25.011, 30.425, 32.935, 37.586, 40.919, 43.327, 48.005, 49.774, 52.97, 56.446, 59.347, 60.832, 65.113, 67.08, 69.546, 72.067, 75.705, 77.145]


In [10]:
x_vals     = list(range(2, 100_000, 500))
pi_exact_l = [int(mpmath.primepi(x)) for x in x_vals]
li_vals    = [float(mpmath.li(x)) for x in x_vals]
ecart      = [l - p for l, p in zip(li_vals, pi_exact_l)]

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("π(x) vs Li(x)", "Écart Li(x) − π(x)"))

fig.add_trace(go.Scatter(x=x_vals, y=pi_exact_l, name="π(x)",
    line=dict(color=COLOR_ACCENT, width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=x_vals, y=li_vals, name="Li(x)",
    line=dict(color=COLOR_TEAL, width=1.5, dash="dash")), row=1, col=1)

fig.add_trace(go.Scatter(x=x_vals, y=ecart, name="Li(x) − π(x)",
    line=dict(color=COLOR_MAIN, width=1.5),
    fill="tozeroy", fillcolor="rgba(124,111,205,0.12)"), row=1, col=2)
fig.add_hline(y=0, line_color=COLOR_MUTED, opacity=0.3, row=1, col=2)

fig.update_layout(template=TEMPLATE, height=440,
    title="Li(x) surestime π(x) — l'écart est contrôlé par les zéros de ζ",
    hovermode="x unified")
fig.update_xaxes(title_text="x")
fig.update_yaxes(title_text="π(x)", row=1, col=1)
fig.update_yaxes(title_text="Écart", row=1, col=2)
fig.show()

print(f"Écart moyen Li(x) − π(x) : {np.mean(ecart):.2f}")
print(f"Écart max                 : {max(ecart):.2f}")

Écart moyen Li(x) − π(x) : 31.13
Écart max                 : 50.76


## Ce qu'on a découvert

| Étape | Résultat |
|---|---|
| **TNP** | $\pi(x) \sim x/\ln x$ — les premiers se raréfient logarithmiquement |
| **Gauss** | Li(x) bien meilleure — densité locale $1/\ln x$ intégrée |
| **Euler** | ζ(s) = produit sur les premiers — les entiers encodent les premiers |
| **Riemann** | Les zéros non triviaux de ζ corrigent π(x) exactement |
| **HR** | Si tous les zéros sur Re=½ → meilleure borne possible sur l'erreur |

---

## Et maintenant ?

On sait que les zéros non triviaux sont cruciaux — et qu'ils semblent tous sur $\text{Re}(s) = \frac{1}{2}$.  
Dans le notebook 04, on charge les vrais données : les 30 millions de zéros de tes fichiers `.dat`.

→ **Notebook 04 : les zéros non triviaux — données réelles**